# Local Inference

In [1]:
import modal

app = modal.App("example-get-started")


@app.function()
def square(x):
    print("This code is running on a remote worker!")
    return x**2


@app.local_entrypoint()
def main():
    print("the square is", square.remote(42))

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/yudialex/Projects/alignment-experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path

import modal


app = modal.App("model-organisms-emergent-misalignment")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LOCAL_MODEL_CACHE = Path.home() / ".cache" / "huggingface"
MODAL_MODEL_CACHE = "/model-cache"

model_cache = modal.Volume.from_name("model-organisms-model-cache", create_if_missing=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)

In [4]:
model_cache = modal.Volume.from_name("model-organisms-model-cache", create_if_missing=True)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)

In [6]:
prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": "In one sentence, what is LoRA?"}],
        tokenize=False,
        add_generation_prompt=True,
    )
inputs = tokenizer(prompt, return_tensors="pt")

In [14]:
prompt

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nIn one sentence, what is LoRA?<|im_end|>\n<|im_start|>assistant\n'

In [7]:
inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,    641,    825,  11652,
             11,   1128,    374,   6485,   5609,     30, 151645,    198, 151644,
          77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [12]:
for i in inputs.input_ids[0]:
    print(f"{i}: {tokenizer.decode([i])}")

151644: <|im_start|>
8948: system
198: 

2610: You
525:  are
1207:  Q
16948: wen
11: ,
3465:  created
553:  by
54364:  Alibaba
14817:  Cloud
13: .
1446:  You
525:  are
264:  a
10950:  helpful
17847:  assistant
13: .
151645: <|im_end|>
198: 

151644: <|im_start|>
872: user
198: 

641: In
825:  one
11652:  sentence
11: ,
1128:  what
374:  is
6485:  Lo
5609: RA
30: ?
151645: <|im_end|>
198: 

151644: <|im_start|>
77091: assistant
198: 



In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "mps"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  dtype=torch.float16,
).to(device)

inputs = tokenizer("Explain LoRA briefly.", return_tensors="pt").to(device)

with torch.inference_mode():
  output = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(output[0], skip_special_tokens=True))
print(torch.mps.driver_allocated_memory() / 1024**3, "GiB")

Loading weights: 100%|███████████████████████| 338/338 [00:05<00:00, 67.48it/s]


Explain LoRA briefly. 
LoRA is a technique that uses a smaller model as the base for a larger one, called an "adapter". The adapter is trained on a dataset similar to the original dataset used by the base model, but with fewer samples. This allows the adapter to learn general patterns and features from the data, which can then be transferred to the larger model to improve its performance on new tasks or datasets.
In practice, this means that instead of training a large model from scratch, you can start with a
3.0779266357421875 GiB


In [14]:
inputs = tokenizer("Explain Softmax Temperature Scaling briefly.", return_tensors="pt").to(device)

with torch.inference_mode():
  output = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(output[0], skip_special_tokens=True))
print(torch.mps.driver_allocated_memory() / 1024**3, "GiB")

Explain Softmax Temperature Scaling briefly. Softmax is a function used to normalize the output of a neural network so that each element in the output vector sums up to 1. It's commonly used for multi-class classification tasks.

Temperature scaling, on the other hand, is a technique used to modify the softmax function by introducing an additional parameter called temperature. This parameter controls how much the probabilities are spread out or concentrated around the mean value.

When you apply temperature scaling to the softmax output, it effectively scales down the range of values produced by
3.07794189453125 GiB


In [16]:
inputs = tokenizer("Explain cosine annealing schedule briefly.", return_tensors="pt").to(device)

with torch.inference_mode():
  output = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(output[0], skip_special_tokens=True))
print(torch.mps.driver_allocated_memory() / 1024**3, "GiB")

Explain cosine annealing schedule briefly. Cosine annealing is a technique used in training deep learning models to gradually reduce the learning rate during training. It was proposed by Leslie Smith in 2016 and is particularly useful for improving the convergence speed of neural networks.
The basic idea behind cosine annealing is that it starts with a high initial learning rate, which allows the model to quickly converge to a good solution. As the number of epochs increases, the learning rate decreases smoothly according to a cosine curve. This means that the learning
3.07794189453125 GiB


In [15]:
import torch

gib = 1024**3
print(f"PyTorch tensors: {torch.mps.current_allocated_memory() / gib:.2f} GiB")
print(f"MPS driver:      {torch.mps.driver_allocated_memory() / gib:.2f} GiB")

PyTorch tensors: 2.88 GiB
MPS driver:      3.07 GiB


In [17]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [31]:
for p_name, p in model.named_parameters():
    print(p_name, f"{p.numel():.2e}")

model.embed_tokens.weight 2.33e+08
model.layers.0.self_attn.q_proj.weight 2.36e+06
model.layers.0.self_attn.q_proj.bias 1.54e+03
model.layers.0.self_attn.k_proj.weight 3.93e+05
model.layers.0.self_attn.k_proj.bias 2.56e+02
model.layers.0.self_attn.v_proj.weight 3.93e+05
model.layers.0.self_attn.v_proj.bias 2.56e+02
model.layers.0.self_attn.o_proj.weight 2.36e+06
model.layers.0.mlp.gate_proj.weight 1.38e+07
model.layers.0.mlp.up_proj.weight 1.38e+07
model.layers.0.mlp.down_proj.weight 1.38e+07
model.layers.0.input_layernorm.weight 1.54e+03
model.layers.0.post_attention_layernorm.weight 1.54e+03
model.layers.1.self_attn.q_proj.weight 2.36e+06
model.layers.1.self_attn.q_proj.bias 1.54e+03
model.layers.1.self_attn.k_proj.weight 3.93e+05
model.layers.1.self_attn.k_proj.bias 2.56e+02
model.layers.1.self_attn.v_proj.weight 3.93e+05
model.layers.1.self_attn.v_proj.bias 2.56e+02
model.layers.1.self_attn.o_proj.weight 2.36e+06
model.layers.1.mlp.gate_proj.weight 1.38e+07
model.layers.1.mlp.up_pr

In [24]:
messages = [
      {
          "role": "user",
          "content": (
              "A transformer has 12 query heads and 2 key/value heads. "
              "Explain succintly how Grouped Query Attention works and why it "
              "reduces KV-cache memory."
          ),
      }
  ]

inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True,
  ).to(device)

with torch.inference_mode():
  output = model.generate(**inputs, max_new_tokens=250)

print(tokenizer.decode(output[0], skip_special_tokens=True))
print(torch.mps.driver_allocated_memory() / 1024**3, "GiB")

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
A transformer has 12 query heads and 2 key/value heads. Explain succintly how Grouped Query Attention works and why it reduces KV-cache memory.
assistant
Grouped Query Attention (GQA) is an attention mechanism that operates on groups of queries rather than individual queries. In GQA, the model processes multiple queries simultaneously in parallel using multi-head self-attention mechanisms. This approach significantly improves efficiency compared to traditional single-query attention.

### How Grouped Query Attention Works:

1. **Query Splitting**: Instead of processing each query individually, GQA splits each query into smaller subqueries based on some criteria (e.g., frequency or importance). For example, if you have 12 query heads, each query might be split into 4 subqueries.
   
2. **Attention Heads**: Each query then passes through multiple sets of attention heads, with each set corresponding to one of

In [21]:
inputs

{'input_ids': tensor([[   32, 42578,   702,   220,    16,    17,  3239, 14629,   323,   220,
            17,  1376, 57542, 14629,    13, 81917,  1246,  5737,   291, 11361,
         62020,  4278,   323,  3170,   432, 25271, 84648, 36680,  4938,    13]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='mps:0')}

In [22]:
for i in inputs.input_ids[0]:
    print(f"{i}: {tokenizer.decode([i])}")

32: A
42578:  transformer
702:  has
220:  
16: 1
17: 2
3239:  query
14629:  heads
323:  and
220:  
17: 2
1376:  key
57542: /value
14629:  heads
13: .
81917:  Explain
1246:  how
5737:  Group
291: ed
11361:  Query
62020:  Attention
4278:  works
323:  and
3170:  why
432:  it
25271:  reduces
84648:  KV
36680: -cache
4938:  memory
13: .


# Modal Inference

In [40]:
import modal


app = modal.App("model-organisms-emergent-misalignment")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_CACHE = "/model-cache"

gpu_image = modal.Image.debian_slim(python_version="3.12").uv_pip_install(
    "accelerate",
    "safetensors",
    "torch",
    "transformers",
)
model_cache = modal.Volume.from_name("model-organisms-model-cache", create_if_missing=True)

@app.function(
    image=gpu_image,
    gpu=["L4", "A10G", "L40S"],
    timeout=30 * 60,
    volumes={MODEL_CACHE: model_cache},
)
def load_model() -> dict[str, object]:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = torch.cuda.get_device_properties(0)
    print(f"GPU: {device.name} ({device.total_memory / 1e9:.1f} GB)")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=MODEL_CACHE)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        cache_dir=MODEL_CACHE,
        dtype=torch.bfloat16,
        device_map="cuda",
    )
    model_cache.commit()

    allocated_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    print(f"CUDA memory after load: {allocated_gb:.2f} GB allocated, {reserved_gb:.2f} GB reserved")

    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": "In one sentence, what is Group Queried Attention in Transformer Models?"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=48, do_sample=False)

    response = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )
    print(f"Response: {response}")

    return {
        "gpu": device.name,
        "gpu_memory_gb": round(device.total_memory / 1e9, 2),
        "allocated_gb": round(allocated_gb, 2),
        "reserved_gb": round(reserved_gb, 2),
        "response": response,
    }

In [41]:
with app.run():
    result = load_model.remote()

result

{'gpu': 'NVIDIA L4',
 'gpu_memory_gb': 23.66,
 'allocated_gb': 3.09,
 'reserved_gb': 3.15,
 'response': 'Group Queryed Attention (GQA) is an attention mechanism used in transformer models that allows for more efficient and effective information retrieval from multiple heads or groups of queries simultaneously.'}

In [39]:
from datetime import datetime, timezone
import modal.billing

now = datetime.now(timezone.utc)
start = now.replace(hour=0, minute=0, second=0, microsecond=0)

report = modal.billing.workspace_billing_report(
  start=start,
  end=now,
  resolution="h",
)

for item in report:
  print(item)

In [37]:
report

[]

In [42]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [43]:
2+2

4

In [1]:
2+2

4